In [1]:
from netmiko import ConnectHandler, BaseConnection

import os
import re
import json

In [2]:
def get_con(ip: str, 
            username: str="root", 
            password: str="cisco", 
            port: int=22, 
            device_type: str="cisco_ios") -> BaseConnection:
    con = ConnectHandler(**{"device_type": device_type, "host": ip, "username": username, "password": password, "port": port})
    con.enable()
    return con

In [3]:
# 명령어를 configure terminal 상태에서 실행시켜주는 함수 -> 실행 결과를 str type으로 리턴
def run_conf(con: BaseConnection, 
             cmd: str) -> str:
    return con.send_config_set(cmd)

In [33]:
def run_cmd(con: BaseConnection, 
            cmd: str) -> str:
    return con.send_command(cmd)


In [5]:
#스위치의 vlan priority 설정
def switch_stp_config(con: BaseConnection,
                       vlan : str,
                       priority : str) -> str:
    
    if (priority % 4096 != 0):
        return "STP 설정 실패 : 잘못된 priority 값:"
 
        
    cmds=[
        f"spanning-tree vlan {vlan} priority {priority}"
    ]
    return run_conf(con, cmds)

In [8]:
con_switch = get_con("192.168.3.130")

In [9]:
switch_stp_config(con_switch, 10, 4096)

'\nESW1#configure terminal\nEnter configuration commands, one per line.  End with CNTL/Z.\nESW1(config)#spanning-tree vlan 10 priority 4096\nESW1(config)#end\nESW1#'

In [10]:
def run_vlan(con: BaseConnection,  
             cmds: str) -> str:

    output = ""
    
    #vlan database 모드 진입
    output += con.send_command("vlan database", expect_string=r"vlan\)")
    
    #cmd
    for c in cmds:
        output += con.send_command(c, expect_string=r"vlan\)")
        
    # 설정을 실제로 반영하고 빠져나오기 
    output += con.send_command("exit", expect_string=r"#")
    
    return output

In [11]:
#스위치의 vlan priority 설정
def add_vlan(con: BaseConnection,
                       vlan : str) -> str:
        
    cmds=[
        f"vlan 30"
    ]
    return run_vlan(con, cmds)

In [12]:
add_vlan(con_switch, 30)

'VLAN 30 added:\n    Name: VLAN0030APPLY completed.\nExiting....'

In [15]:
con_r = get_con("172.16.124.1")

In [21]:
def inter_vlan_config(con: BaseConnection,
                      interface: str,
                      vlan: str,
                      ip: str,
                      netmask: str) -> str:
    cmds = [
        f"int {interface}.{vlan}",
        f"enc dot1Q {vlan}",
        f"ip add {ip} {netmask}",
        "exit",
        f"int {interface}",
        "no sh"
    ]
    return run_conf(con, cmds)

In [22]:
inter_vlan_config(con_r, "f0/0", "30", "1.1.1.0", "255.255.255.0")

'configure terminal\nEnter configuration commands, one per line.  End with CNTL/Z.\nR1(config)#int f0/0.30\nR1(config-subif)#enc dot1Q 30\nR1(config-subif)#ip add 1.1.1.0 255.255.255.0\nBad mask /24 for address 1.1.1.0\nR1(config-subif)#exit\nR1(config)#int f0/0\nR1(config-if)#no sh\nR1(config-if)#end\nR1#'

In [23]:
def switch_vtp_config(con: BaseConnection,
                      domain_name: str,
                      password: str,
                      vtp_mode: str = "Server"
                     ) -> str:
    cmds = [
        f"vtp {vtp_mode}",
        f"vtp domain {domain_name}",
        f"vtp password {password}"
    ]
    return run_vlan(con, cmds)

In [26]:
con_switch4 = get_con("192.168.3.134")

In [27]:
switch_vtp_config(con_switch4, "k.com", "cisco", "transparent")

'Device mode already VTP TRANSPARENT.Changing VTP domain name from kong.com to k.comPassword already set to cisco.APPLY completed.\nExiting....'

In [37]:
#스위치 정보 
def get_stp_info(con: BaseConnection) -> str:
    cmd= "sh spanning-tree bri" 
    return run_cmd(con, cmd)

In [38]:
get_stp_info(con_switch4)

'\nVLAN1\n  Spanning tree enabled protocol ieee\n  Root ID    Priority    32768\n             Address     c403.2598.0000\n             Cost        38\n             Port        41 (FastEthernet1/0)\n             Hello Time   2 sec  Max Age 20 sec  Forward Delay 15 sec\n\n  Bridge ID  Priority    32768\n             Address     c405.2ed8.0000\n             Hello Time   2 sec  Max Age 20 sec  Forward Delay 15 sec\n             Aging Time 300\n\nInterface                                   Designated\nName                 Port ID Prio Cost  Sts Cost  Bridge ID            Port ID\n-------------------- ------- ---- ----- --- ----- -------------------- -------\nFastEthernet1/0      128.41   128    19 FWD    19 32768 c404.2fb4.0000 128.42 \nFastEthernet1/1      128.42   128    19 FWD    38 32768 c405.2ed8.0000 128.42 \n\n\nVLAN10\n  Spanning tree enabled protocol ieee\n  Root ID    Priority    4096\n             Address     c401.2c10.0001\n             Cost        76\n             Port        4

In [39]:
#inter vlan 
def get_vlan_interfaces(con: BaseConnection) -> dict:

    # 라우터에서 sh run 명령어 실행
    raw_config = con.send_command("show running-config")
    
    router_name = "Unknown_Router" #default name
    router_backup = {}
    
    # \n 를 기준으로 전체 텍스트 쪼개기
    lines = raw_config.splitlines()
    
    # 먼저 첫 번째 패스로 라우터 이름(hostname)부터 찾기
    for line in lines:
        line = line.strip()
        if line.startswith("hostname "):
            router_name = line.split()[1] # "hostname R1" 에서 "R1"만 쏙 가져옴
            break
            
    # 라우터 이름을 키로 가진 딕셔너리 리스트 생성
    router_backup[router_name] = []
    
    # 3. 가상 인터페이스 및 IP 정보 추적 시작
    current_iface = None
    vlan_id = None
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
            
        # 가상 인터페이스 구역 진입 확인 (예: interface FastEthernet0/0.10)
        if line.startswith("interface ") and "." in line:
            current_iface = line.split()[1]           # 두 번째 단어인 인터페이스명 가져오기
            vlan_id = current_iface.split(".")[-1]      # 소수점 뒷자리를 VLAN ID로 지정
            
            # 딕셔너리 뼈대 생성해서 넣어두기
            iface_info = {
                "interface": current_iface,
                "vlan": vlan_id,
                "ip": "",
                "netmask": ""
            }
            router_backup[router_name].append(iface_info)
            continue
            
        # 일반 물리 포트 인터페이스를 만나면 반복문으로 돌아가기
        elif line.startswith("interface ") and "." not in line:
            current_iface = None
            vlan_id = None
            continue
            
        # 가상 포트 구역 내부  +  ip address 라인을 만났을 때
        if current_iface and line.startswith("ip address "):
            parts = line.split()
            # parts 예시: ['ip', 'address', '10.0.10.1', '255.255.255.0']
            if len(parts) >= 4:
                ip_addr = parts[2]
                netmask_addr = parts[3]
                
                # 가장 마지막에 리스트에 추가된 가상 포트 딕셔너리에 데이터 채우기
                router_backup[router_name][-1]["ip"] = ip_addr
                router_backup[router_name][-1]["netmask"] = netmask_addr
                
    return router_backup

In [40]:
def store_vlan_interfaces(con: BaseConnection, 
                      vlan_int_setting_json_path: str) -> dict:
    """
    현재 라우터의 NAT 설정을 읽어 JSON 파일에 저장
    """
    # 1. 현재 라우터의 NAT 설정 정보 가져오기
    vlan_int_config = get_vlan_interfaces(con)
    
    # 2. 데이터 구조 초기화 및 기존 파일 읽기
    data = {}
    if os.path.exists(vlan_int_setting_json_path) and os.path.getsize(vlan_int_setting_json_path) > 0:
        with open(vlan_int_setting_json_path, 'r', encoding="utf-8") as f:
            data = json.load(f)

    # 3. 데이터 업데이트 (장비 호스트명을 키로 저장)
    data.update(vlan_int_config)

    # 4. 파일 저장
    with open(vlan_int_setting_json_path, 'w', encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    
    return vlan_int_config

In [42]:
con_r = get_con("172.16.124.1")

In [43]:
store_vlan_interfaces(con_r, "router_vlan_int.txt")

{'R1': [{'interface': 'Ethernet2/2.10',
   'vlan': '10',
   'ip': '192.168.3.129',
   'netmask': '255.255.255.192'},
  {'interface': 'Ethernet2/2.20',
   'vlan': '20',
   'ip': '192.168.3.193',
   'netmask': '255.255.255.192'}]}

In [44]:
#static nat 설정
def router_static_nat_config(con: BaseConnection,
                      in_interface: str,
                      out_interface: str,
                      in_addr: str,
                      out_addr: str) -> str:
    cmds = [
        f"interface {in_interface}",
        "ip nat inside",
        f"interface {out_interface}",
        "ip nat outside",
        f"ip nat inside source static {in_addr} {out_addr}"
    ]
    return run_conf(con, cmds)

In [51]:
con_r = get_con("172.16.124.1")

NetmikoTimeoutException: TCP connection to device failed.

Common causes of this problem are:
1. Incorrect hostname or IP address.
2. Wrong TCP port.
3. Intermediate firewall blocking access.

Device settings: cisco_ios 172.16.124.1:22



In [48]:
router_static_nat_config(con_r, "e2/0", "f0/0", "192.168.3.10", "172.16.124.1")

ReadTimeout: 

Pattern not detected: '[>#]' in output.

Things you might try to fix this:
1. Adjust the regex pattern to better identify the terminating string. Note, in
many situations the pattern is automatically based on the network device's prompt.
2. Increase the read_timeout to a larger value.

You can also look at the Netmiko session_log or debug log for more information.



In [58]:
import paramiko

ip = "172.16.10.10"
user = "root"
pw = "asd123!@"

def ssh(ip, user, pw, cmd):
    cli = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(ip, username=user, password=pw, port=22)
    (stdin, stdout, stderr) = cli.exec_command(cmd)
    result = stdout.readlines()
    cli.close()
    return result

# SELinux 확인
cmd = "getenforce && sestatus"
print("=" * 40)
print("[ SELinux 상태 ]")
print("=" * 40)
for line in ssh(ip, user, pw, cmd):
    print(line.strip())

# firewalld 서비스 상태
cmd = "systemctl status firewalld --no-pager"
print()
print("=" * 40)
print("[ Firewalld 서비스 상태 ]")
print("=" * 40)
for line in ssh(ip, user, pw, cmd):
    print(line.strip())

# 기본 존
cmd = "firewall-cmd --get-default-zone"
print()
print("[ 기본 존 ]")
for line in ssh(ip, user, pw, cmd):
    print(line.strip())

# 허용된 서비스 및 포트
cmd = "firewall-cmd --list-all"
print()
print("[ 허용 서비스 / 포트 ]")
for line in ssh(ip, user, pw, cmd):
    print(line.strip())

[ SELinux 상태 ]
Disabled
SELinux status:                 disabled

[ Firewalld 서비스 상태 ]
○ firewalld.service - firewalld - dynamic firewall daemon
Loaded: loaded (/usr/lib/systemd/system/firewalld.service; disabled; preset: enabled)
Active: inactive (dead)
Docs: man:firewalld(1)

[ 기본 존 ]

[ 허용 서비스 / 포트 ]


In [56]:
import paramiko

ip = "172.16.5.112"
user = "root"
pw = "asd123!@"

def ssh(ip, user, pw, cmd):
    cli = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(ip, username=user, password=pw, port=22)
    (stdin, stdout, stderr) = cli.exec_command(cmd)
    result = stdout.readlines()
    cli.close()
    return result

# SELinux 확인
cmd = "getenforce && sestatus"
print("=" * 40)
print("[ SELinux 상태 ]")
print("=" * 40)
for line in ssh(ip, user, pw, cmd):
    print(line.strip())

# firewalld 서비스 상태
cmd = "systemctl status firewalld"
print()
print("=" * 40)
print("[ Firewalld 서비스 상태 ]")
print("=" * 40)
for line in ssh(ip, user, pw, cmd):
    print(line.strip())

# 사용중인  포트
cmd = "firewall-cmd --list-all"
print()
print("[ 허용 서비스 / 포트 ]")
for line in ssh(ip, user, pw, cmd):
    print(line.strip())

[ SELinux 상태 ]
Enforcing
SELinux status:                 enabled
SELinuxfs mount:                /sys/fs/selinux
SELinux root directory:         /etc/selinux
Loaded policy name:             targeted
Current mode:                   enforcing
Mode from config file:          enforcing
Policy MLS status:              enabled
Policy deny_unknown status:     allowed
Memory protection checking:     actual (secure)
Max kernel policy version:      33

[ Firewalld 서비스 상태 ]
● firewalld.service - firewalld - dynamic firewall daemon
Loaded: loaded (/usr/lib/systemd/system/firewalld.service; enabled; preset: enabled)
Active: active (running) since Tue 2026-05-19 15:52:40 KST; 12min ago
Docs: man:firewalld(1)
Process: 830 ExecStartPost=/usr/bin/firewall-cmd --state (code=exited, status=0/SUCCESS)
Main PID: 815 (firewalld)
Tasks: 2 (limit: 16851)
Memory: 44.4M (peak: 64.2M)
CPU: 1.320s
CGroup: /system.slice/firewalld.service
└─815 /usr/bin/python3 -s /usr/sbin/firewalld --nofork --nopid

5월 19 15:52:3

In [59]:

# 사용중인 포트 확인
cmd = "ss -ntlp"
print("=" * 40)
print("[ 사용 중인 포트 ]")
print("=" * 40)
for line in ssh(ip, user, pw, cmd):
    print(line.strip())


[ 사용 중인 포트 ]
State  Recv-Q Send-Q Local Address:Port Peer Address:PortProcess
LISTEN 0      128          0.0.0.0:22        0.0.0.0:*    users:(("sshd",pid=838,fd=3))
LISTEN 0      128          0.0.0.0:8080      0.0.0.0:*    users:(("jupyter-noteboo",pid=836,fd=6))
LISTEN 0      511                *:80              *:*    users:(("httpd",pid=71306,fd=4),("httpd",pid=70752,fd=4),("httpd",pid=70751,fd=4),("httpd",pid=70750,fd=4),("httpd",pid=12653,fd=4))
LISTEN 0      128             [::]:22           [::]:*    users:(("sshd",pid=838,fd=4))
